In [86]:
import pandas as pd
import numpy as np
import importlib
import gc
import sys
import warnings
sys.path.append('./scripts')  
import preprocesamiento
import target
import feature_engineering
import model_lgb
import model_lgb_simple
importlib.reload(preprocesamiento)
importlib.reload(target)
importlib.reload(model_lgb)
importlib.reload(model_lgb_simple)
importlib.reload(feature_engineering)
warnings.filterwarnings("ignore")

# Experimento 9: 
- LGBM
- Estandarización hasta 201906
- Estandarizacion del target
- Semillerio en train y test
- Agrego variables: agrego los ceros que dijo el profesor.
- Pesos: logaritmo
- sqlite:///optuna_studies_v22.db
- Kaggle =  


**Training:**
```python
training = [
    201701, 201702, 201703, 201704, 201705, 201706, 201707, 201708, 201709,
    201710, 201711, 201712, 201801, 201802, 201803, 201804, 201805,
    201806, 201807, 201808, 201809, 201810, 201811, 201812,
    201901, 201902, 201903, 201904, 201905, 201906
]

validation = [
    # 201907, 201908
    201907, 201909
]

testing = [
    201910
]


Levantamos

In [ ]:
df = pd.read_csv('./datasets/periodo_x_producto.csv', sep=',', encoding='utf-8')

Estandarizacion zscore

In [48]:
df = preprocesamiento.normalizar_con_zscore(df, "tn", fecha=201806)
df

,product_id,periodo,nacimiento_producto,muerte_producto,mes_n,total_meses,producto_nuevo,ciclo_de_vida_inicial,cat1,cat2,...,brand,sku_size,stock_final,tn,plan_precios_cuidados,cust_request_qty,cust_request_tn,tn_mean,tn_std,tn_zscore
0,20001,201701,201701,201912,1,36,0,0,HC,ROPA LAVADO,...,ARIEL,3000.0,NaN,934.77222,0.0,479.0,937.72717,1254.369806,251.944810,-1.268522
1,20001,201702,201701,201912,2,36,0,0,HC,ROPA LAVADO,...,ARIEL,3000.0,NaN,798.01620,0.0,432.0,833.72187,1254.369806,251.944810,-1.811324
2,20001,201703,201701,201912,3,36,0,0,HC,ROPA LAVADO,...,ARIEL,3000.0,NaN,1303.35771,0.0,509.0,1330.74697,1254.369806,251.944810,0.194439
3,20001,201704,201701,201912,4,36,0,0,HC,ROPA LAVADO,...,ARIEL,3000.0,NaN,1069.96130,0.0,279.0,1132.94430,1254.369806,251.944810,-0.731940
4,20001,201705,201701,201912,5,36,0,0,HC,ROPA LAVADO,...,ARIEL,3000.0,NaN,1502.20132,0.0,701.0,1550.68936,1254.369806,251.944810,0.983674
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
31357,21281,201704,201702,201708,3,7,1,1,NaN,NaN,...,NaN,NaN,NaN,0.00000,NaN,NaN,NaN,0.021060,0.031414,-0.670410
31358,21281,201705,201702,201708,4,7,1,0,NaN,NaN,...,NaN,NaN,NaN,0.00000,NaN,NaN,NaN,0.021060,0.031414,-0.670410
31359,21281,201706,201702,201708,5,7,1,0,NaN,NaN,...,NaN,NaN,NaN,0.09134,0.0,8.0,0.10539,0.021060,0.031414,2.237248
31360,21281,201707,201702,201708,6,7,1,0,NaN,NaN,...,NaN,NaN,NaN,0.00000,NaN,NaN,NaN,0.021060,0.031414,-0.670410


In [49]:
df.rename(columns={'tn':'tn_original'}, inplace=True)
df.rename(columns={'tn_zscore':'tn'}, inplace=True)

Guardamos

In [12]:
df.to_csv("../../data/preprocessed/periodo_x_producto_con_target_zscore_201906.csv", index=False, sep=',', encoding='utf-8')

##### Procesamiento del Target

In [50]:
df = pd.read_csv("../../data/preprocessed/periodo_x_producto_con_target_zscore_201906.csv", sep=',', encoding='utf-8')

print(df.shape)

df = target.target_tn_mas_dos(df)

print(df.shape)

(31362, 21)
(31362, 22)


##### Feature Engineering

In [51]:
columnas_baseline = df.columns.tolist()
columnas_baseline

['product_id',
 'periodo',
 'nacimiento_producto',
 'muerte_producto',
 'mes_n',
 'total_meses',
 'producto_nuevo',
 'ciclo_de_vida_inicial',
 'cat1',
 'cat2',
 'cat3',
 'brand',
 'sku_size',
 'stock_final',
 'tn_original',
 'plan_precios_cuidados',
 'cust_request_qty',
 'cust_request_tn',
 'tn_mean',
 'tn_std',
 'tn',
 'target']

##### Preprocesamiento a la minima expresión :)

In [52]:
df = feature_engineering.create_category_features_cat1(df)
df = feature_engineering.create_category_features_cat2(df)
df = feature_engineering.create_category_features_cat3(df)

In [53]:
# ##### aplicamos OHE
df = preprocesamiento.aplicarOHE(df)
df.shape

(31362, 174)

### Feature Engineering

##### Neural Prophet

In [54]:
neural_prophet_fe = pd.read_csv("./datasets/features_neuralprophet_completo.csv", sep=',', encoding='utf-8')
neural_prophet_fe['ds'] = pd.to_datetime(neural_prophet_fe['ds'], errors='coerce')
# Versión alternativa más robusta:
neural_prophet_fe['periodo'] = neural_prophet_fe['ds'].apply(
    lambda x: x.year * 100 + x.month if pd.notnull(x) else None
)
neural_prophet_fe = neural_prophet_fe[['periodo', 'product_id', 'trend', "season_yearly", "season_monthly"]]
df = df.merge(neural_prophet_fe, on=['periodo', 'product_id'], how='left')
df.shape

(31362, 177)

##### Prophet

In [55]:
prophet_fe = pd.read_csv("./datasets/prophet_features_tn_zscore.csv", sep=',', encoding='utf-8')
prophet_fe['ds'] = pd.to_datetime(prophet_fe['ds'], errors='coerce')
prophet_fe['periodo'] = prophet_fe['ds'].apply(
    lambda x: x.year * 100 + x.month if pd.notnull(x) else None
)
prophet_fe = prophet_fe[['periodo', 'product_id', 'trend_add', "yearly_add", "additive_terms", 'trend_mult', 'yearly_mult', 'multiplicative_terms']]
df = df.merge(prophet_fe, on=['periodo', 'product_id'], how='left')
df.shape

(31362, 183)

##### FE Moviles

In [56]:
df = feature_engineering.get_lags(df, "tn", 201912)
df = feature_engineering.get_delta_lags(df, "tn", 24)
df = feature_engineering.get_rolling_means(df, "tn", 201912)
df = feature_engineering.get_rolling_stds(df, "tn", 201912)
df = feature_engineering.get_rolling_mins(df, "tn", 201912)
df = feature_engineering.get_rolling_maxs(df, "tn", 201912)
df = feature_engineering.get_rolling_medians(df, "tn", 201912)
df = feature_engineering.get_rolling_skewness(df, "tn", 201912)
df = feature_engineering.get_autocorrelaciones(df, "tn", 201912)
df.shape

(31362, 745)

In [57]:
df = feature_engineering.get_lags(df, "cust_request_qty", 201912)
df = feature_engineering.get_delta_lags(df, "cust_request_qty", 24)
df = feature_engineering.get_rolling_means(df, "cust_request_qty", 201912)
df = feature_engineering.get_rolling_stds(df, "cust_request_qty", 201912)
df = feature_engineering.get_rolling_mins(df, "cust_request_qty", 201912)
df = feature_engineering.get_rolling_maxs(df, "cust_request_qty", 201912)
df.shape

(31362, 1200)

In [58]:
df = feature_engineering.get_lags(df, "stock_final", 201912)
df = feature_engineering.get_delta_lags(df, "stock_final", 24)
df = feature_engineering.get_rolling_means(df, "stock_final", 201912)
df = feature_engineering.get_rolling_stds(df, "stock_final", 201912)
df = feature_engineering.get_rolling_mins(df, "stock_final", 201912)
df = feature_engineering.get_rolling_maxs(df, "stock_final", 201912)
df.shape

(31362, 1655)

Features Diana

In [59]:
df = feature_engineering.calcular_diferencia_con_medias_moviles(df)
df = feature_engineering.calcular_ratios_con_medias_moviles(df)
df.shape

(31362, 1691)

##### FE Moviles sobre otras variables

In [56]:
# #  stock final
# df = feature_engineering.get_lagsEspecificos(df, col='stock_final_zscore')
# df = feature_engineering.get_delta_lags_especificos(df, col='stock_final_zscore')
# df = feature_engineering.get_rolling_means_especificos(df, col='stock_final_zscore')
# df = feature_engineering.get_rolling_stds_especificos(df, col='stock_final_zscore')
# df = feature_engineering.get_rolling_medians_especificos(df, col='stock_final_zscore')
# df = feature_engineering.get_rolling_mins_especificos(df, col='stock_final_zscore')
# df = feature_engineering.get_rolling_maxs_especificos(df, col='stock_final_zscore')
# df = feature_engineering.get_rolling_skewness_especificos(df, col='stock_final_zscore')

#  cust_request_qty
# df = feature_engineering.get_lagsEspecificos(df, col='cust_request_qty')
# df = feature_engineering.get_delta_lags_especificos(df, col='cust_request_qty')
# df = feature_engineering.get_rolling_means_especificos(df, col='cust_request_qty')
# df = feature_engineering.get_rolling_stds_especificos(df, col='cust_request_qty')
# df = feature_engineering.get_rolling_mins_especificos(df, col='cust_request_qty')
# df = feature_engineering.get_rolling_maxs_especificos(df, col='cust_request_qty')
# df = feature_engineering.get_rolling_medians_especificos(df, col='cust_request_qty')
# df = feature_engineering.get_rolling_skewness_especificos(df, col='cust_request_qty')

##### FE Calendario

In [60]:
df = feature_engineering.generar_ids(df)
df = feature_engineering.get_componentesTemporales(df)
df = feature_engineering.get_anomaliasPoliticas(df)
# df = feature_engineering.descomposicion_serie_temporal(df, col='tn')
df.shape

(31362, 1716)

##### FE sobre FE

In [61]:
df = feature_engineering.chatGPT_features_serie(df, "tn")
df = feature_engineering.mes_con_feriado(df)
df.shape

(31362, 1745)

##### Variables Exogenas

In [62]:
df = feature_engineering.get_dolar(df)
df = feature_engineering.get_IPC(df)
df['ipc'] = df['ipc'].str.replace(',', '.').astype(float)
df['dolar'] = df['dolar'].str.replace(',', '.').astype(float)
# df.drop(columns=['ds'], inplace=True)
df.fillna(0, inplace=True) ##### EXPERIMENTAR
df = feature_engineering.correlacion_exogenas(df)
df = feature_engineering.get_mes_receso_escolar(df)
df.shape

(31362, 1748)

##### Nuevas FE

In [63]:
df = feature_engineering.create_ratio_features(df)
df = feature_engineering.enhance_lifecycle_features(df)
# df = feature_engineering.create_category_features(df)
df = feature_engineering.create_regime_features(df)
df = feature_engineering.create_nonlinear_trends(df)
df = feature_engineering.create_temporal_interactions(df)
df = feature_engineering.create_asymmetric_window_features(df)
df = feature_engineering.recomendaciones_deepseek(df)
df = feature_engineering.get_nuevas_features(df)
df.shape

(31362, 1787)

##### Ceros

In [17]:
df = feature_engineering.agregar_ceros_consecutivos_atras(df, col_tn='tn_original')
df = feature_engineering.agregar_no_ceros_consecutivos_atras(df, col_tn='tn_original')
df = feature_engineering.agregar_ceros_ultimos_n_meses(df, ventanas=[1,2,3,4,5,6,12], col_tn='tn_original')
df = feature_engineering.agregar_min_max_ult_n(df, n_list=(1,2,3,4,5,6,12), col_tn='tn_original')
df.shape

(31362, 1810)

##### Elimino aquellas que no sirven

In [ ]:
import json
import pandas as pd
import csv

with open("./feature_importance/v19.json") as f:
    data = json.load(f)

# Crear una lista de tuplas (feature, value)
features_values = [(feature, value) for feature, value in data.items()]

# Guardar en un archivo CSV
with open('./feature_importance/v19.csv', 'w', newline='') as csvfile:
    writer = csv.writer(csvfile)
    writer.writerow(['feature', 'importance'])  # Escribir el encabezado
    writer.writerows(features_values)      # Escribir los datos

print("Archivo CSV generado exitosamente: features_values.csv")

Archivo CSV generado exitosamente: features_values.csv


In [40]:
importantes = pd.read_csv("./feature_importance/v19.csv", sep=',', encoding='utf-8')
no_importantes = importantes[importantes['importance'] == 0]
no_importantes = no_importantes[~no_importantes['feature'].isin(columnas_baseline)]
no_importantes

,feature,importance
1103,tn_rolling_std_20,0.0
1104,tn_rolling_std_22,0.0
1105,tn_rolling_std_25,0.0
1106,tn_rolling_std_26,0.0
1107,tn_rolling_std_27,0.0
...,...,...
1781,cat3_Acond Bebe,0.0
1782,tn_rolling_median_25,0.0
1783,tn_rolling_median_24,0.0
1786,tn_rolling_std_1,0.0


In [41]:
cols_a_eliminar = no_importantes.feature.unique()
print(f"Antes de eliminar: {df.shape[1]} columnas")
df = df.drop(columns=cols_a_eliminar, errors='ignore')
print(f"Después de eliminar: {df.shape[1]} columnas")

Antes de eliminar: 1787 columnas
Después de eliminar: 1104 columnas


Eliminar object/categorical columnas

In [64]:
df = df.select_dtypes(exclude=['datetime', 'datetime64', 'object', 'category'])

In [45]:
df['target'].isna().sum()

np.int64(0)

Train Test Split

In [65]:
train = df[df['periodo'] <= 201912]
test = df[df['periodo'] == 201912]

Entrenamiento

In [68]:

model_lgb_simple.optimizar_con_optuna_sin5FCV_con_semillerio_db(train, version="v22", n_trials=500)


Para visualizar los resultados en tiempo real:
1. Abre otra terminal y ejecuta:
   optuna-dashboard sqlite:///optuna_studies_v22.db
2. Abre en tu navegador: http://127.0.0.1:8080/


[I 2025-07-13 19:55:28,378] Using an existing study with name 'lightgbm_optimization_v22' instead of creating a new one.
[I 2025-07-13 19:56:36,412] Trial 289 finished with value: 124.27045080936986 and parameters: {'num_leaves': 83, 'learning_rate': 0.10809366764261591, 'feature_fraction': 0.6668086637168998, 'bagging_fraction': 0.9697017443698029, 'bagging_freq': 8, 'lambda_l1': 5.5313912737064595e-06, 'lambda_l2': 0.030070928012921057, 'min_child_samples': 49, 'max_depth': 10, 'max_bin': 322, 'min_data_in_leaf': 35, 'extra_trees': True, 'early_stopping_rounds': 23, 'path_smooth': 0.4478078172035074, 'min_gain_to_split': 0.047899714155294044}. Best is trial 257 with value: 0.44357105642948386.


Mejor trial hasta ahora: RMSE=0.443571, Parámetros={'num_leaves': 24, 'learning_rate': 0.08756616330710833, 'feature_fraction': 0.6330071641441583, 'bagging_fraction': 0.9678808037613486, 'bagging_freq': 10, 'lambda_l1': 1.1634123907122364e-08, 'lambda_l2': 0.047286936432928375, 'min_child_samples': 43, 'max_depth': 10, 'max_bin': 479, 'min_data_in_leaf': 37, 'extra_trees': True, 'early_stopping_rounds': 29, 'path_smooth': 0.9996568294482207, 'min_gain_to_split': 0.03491779707899573}


[I 2025-07-13 19:58:41,638] Trial 290 finished with value: 123.51206184111388 and parameters: {'num_leaves': 16, 'learning_rate': 0.09846709011591549, 'feature_fraction': 0.6342974086889194, 'bagging_fraction': 0.9931336989753181, 'bagging_freq': 10, 'lambda_l1': 9.030211108439333e-07, 'lambda_l2': 1.7270812116459777e-08, 'min_child_samples': 47, 'max_depth': 10, 'max_bin': 486, 'min_data_in_leaf': 67, 'extra_trees': False, 'early_stopping_rounds': 33, 'path_smooth': 0.8469961233886656, 'min_gain_to_split': 0.06064486906058383}. Best is trial 257 with value: 0.44357105642948386.


Mejor trial hasta ahora: RMSE=0.443571, Parámetros={'num_leaves': 24, 'learning_rate': 0.08756616330710833, 'feature_fraction': 0.6330071641441583, 'bagging_fraction': 0.9678808037613486, 'bagging_freq': 10, 'lambda_l1': 1.1634123907122364e-08, 'lambda_l2': 0.047286936432928375, 'min_child_samples': 43, 'max_depth': 10, 'max_bin': 479, 'min_data_in_leaf': 37, 'extra_trees': True, 'early_stopping_rounds': 29, 'path_smooth': 0.9996568294482207, 'min_gain_to_split': 0.03491779707899573}


...

[I 2025-07-14 03:22:19,487] Trial 787 finished with value: 111.0677084003574 and parameters: {'num_leaves': 21, 'learning_rate': 0.117427100422794, 'feature_fraction': 0.6361322283763446, 'bagging_fraction': 0.9739571567914658, 'bagging_freq': 10, 'lambda_l1': 2.209430554948189e-08, 'lambda_l2': 0.05323831387671234, 'min_child_samples': 45, 'max_depth': 10, 'max_bin': 360, 'min_data_in_leaf': 30, 'extra_trees': True, 'early_stopping_rounds': 25, 'path_smooth': 0.9632268878294241, 'min_gain_to_split': 0.04781117627974818}. Best is trial 257 with value: 0.44357105642948386.


Mejor trial hasta ahora: RMSE=0.443571, Parámetros={'num_leaves': 24, 'learning_rate': 0.08756616330710833, 'feature_fraction': 0.6330071641441583, 'bagging_fraction': 0.9678808037613486, 'bagging_freq': 10, 'lambda_l1': 1.1634123907122364e-08, 'lambda_l2': 0.047286936432928375, 'min_child_samples': 43, 'max_depth': 10, 'max_bin': 479, 'min_data_in_leaf': 37, 'extra_trees': True, 'early_stopping_rounds': 29, 'path_smooth': 0.9996568294482207, 'min_gain_to_split': 0.03491779707899573}


[I 2025-07-14 03:23:04,615] Trial 788 finished with value: 111.29473651694619 and parameters: {'num_leaves': 53, 'learning_rate': 0.13029234087438643, 'feature_fraction': 0.6717772540370822, 'bagging_fraction': 0.8887508440038063, 'bagging_freq': 10, 'lambda_l1': 4.124916386111299e-06, 'lambda_l2': 0.15850050323163212, 'min_child_samples': 50, 'max_depth': 10, 'max_bin': 451, 'min_data_in_leaf': 37, 'extra_trees': True, 'early_stopping_rounds': 27, 'path_smooth': 0.9436288910461358, 'min_gain_to_split': 0.31760807751623843}. Best is trial 257 with value: 0.44357105642948386.


Mejor trial hasta ahora: RMSE=0.443571, Parámetros={'num_leaves': 24, 'learning_rate': 0.08756616330710833, 'feature_fraction': 0.6330071641441583, 'bagging_fraction': 0.9678808037613486, 'bagging_freq': 10, 'lambda_l1': 1.1634123907122364e-08, 'lambda_l2': 0.047286936432928375, 'min_child_samples': 43, 'max_depth': 10, 'max_bin': 479, 'min_data_in_leaf': 37, 'extra_trees': True, 'early_stopping_rounds': 29, 'path_smooth': 0.9996568294482207, 'min_gain_to_split': 0.03491779707899573}
Estudio guardado en: sqlite:///optuna_studies_v22.db

Mejores hiperparámetros encontrados:
num_leaves: 24
learning_rate: 0.08756616330710833
feature_fraction: 0.6330071641441583
bagging_fraction: 0.9678808037613486
bagging_freq: 10
lambda_l1: 1.1634123907122364e-08
lambda_l2: 0.047286936432928375
min_child_samples: 43
max_depth: 10
max_bin: 479
min_data_in_leaf: 37
extra_trees: True
early_stopping_rounds: 29
path_smooth: 0.9996568294482207
min_gain_to_split: 0.03491779707899573


(<optuna.study.study.Study at 0x1cf87870210>,
 {'num_leaves': 24,
  'learning_rate': 0.08756616330710833,
  'feature_fraction': 0.6330071641441583,
  'bagging_fraction': 0.9678808037613486,
  'bagging_freq': 10,
  'lambda_l1': 1.1634123907122364e-08,
  'lambda_l2': 0.047286936432928375,
  'min_child_samples': 43,
  'max_depth': 10,
  'max_bin': 479,
  'min_data_in_leaf': 37,
  'extra_trees': True,
  'early_stopping_rounds': 29,
  'path_smooth': 0.9996568294482207,
  'min_gain_to_split': 0.03491779707899573,
  'objective': 'regression',
  'metric': 'rmse',
  'boosting_type': 'gbdt',
  'verbosity': -1})

Prediccion

In [71]:
df_future = model_lgb_simple.semillerio_en_prediccion_con_pesos(train, test, version="v22")

In [72]:
df_future

,periodo,product_id,target,pred
30476,201912,20001,0.0,0.744316
30477,201912,20002,0.0,0.353683
30478,201912,20003,0.0,-0.050623
30479,201912,20004,0.0,0.743757
30480,201912,20005,0.0,0.510273
...,...,...,...,...
31357,201912,21265,0.0,1.267023
31358,201912,21266,0.0,-0.337602
31359,201912,21267,0.0,-5.911647
31360,201912,21271,0.0,-0.513513


Filtramos los 180 productos

In [76]:
productos_ok = pd.read_csv("../../data/raw/product_id_apredecir201912.csv", sep="\t")
df_future = df_future[df_future['periodo'] == 201912]
df_future = df_future[df_future['product_id'].isin(productos_ok['product_id'].unique())]


In [78]:
df_future

,periodo,product_id,target,pred
30476,201912,20001,0.0,0.744316
30477,201912,20002,0.0,0.353683
30478,201912,20003,0.0,-0.050623
30479,201912,20004,0.0,0.743757
30480,201912,20005,0.0,0.510273
...,...,...,...,...
31355,201912,21263,0.0,-0.717058
31357,201912,21265,0.0,1.267023
31358,201912,21266,0.0,-0.337602
31359,201912,21267,0.0,-5.911647


In [79]:
df_future_copy = df_future.copy()

In [80]:
import os
ruta_archivo = f'./datasets/tn_stats_201906.csv'
    
df_stats = pd.DataFrame()

if os.path.exists(ruta_archivo) and ruta_archivo.endswith('.csv'):
    df_stats = pd.read_csv(ruta_archivo, sep=',')

df_stats

,product_id,tn_mean,tn_std
0,20001,1375.882837,310.489330
1,20002,962.547075,258.093361
2,20003,892.114038,307.871182
3,20004,665.386502,225.992372
4,20005,627.744458,220.138757
...,...,...,...
1111,21271,0.026155,0.020938
1112,21273,0.057242,0.124272
1113,21274,0.067028,0.096980
1114,21276,0.089478,0.030062


In [81]:
df_future_copy = df_future_copy.merge(df_stats, on=['product_id'], how='left')
df_future_copy

,periodo,product_id,target,pred,tn_mean,tn_std
0,201912,20001,0.0,0.744316,1375.882837,310.489330
1,201912,20002,0.0,0.353683,962.547075,258.093361
2,201912,20003,0.0,-0.050623,892.114038,307.871182
3,201912,20004,0.0,0.743757,665.386502,225.992372
4,201912,20005,0.0,0.510273,627.744458,220.138757
...,...,...,...,...,...,...
775,201912,21263,0.0,-0.717058,0.133198,0.178028
776,201912,21265,0.0,1.267023,0.151885,0.136228
777,201912,21266,0.0,-0.337602,0.151885,0.134848
778,201912,21267,0.0,-5.911647,0.160505,0.073935


In [82]:

df_future_copy['tn'] = df_future_copy['pred'] * df_future_copy['tn_std'] + df_future_copy['tn_mean']
df_future_copy

,periodo,product_id,target,pred,tn_mean,tn_std,tn
0,201912,20001,0.0,0.744316,1375.882837,310.489330,1606.984862
1,201912,20002,0.0,0.353683,962.547075,258.093361,1053.830220
2,201912,20003,0.0,-0.050623,892.114038,307.871182,876.528653
3,201912,20004,0.0,0.743757,665.386502,225.992372,833.469882
4,201912,20005,0.0,0.510273,627.744458,220.138757,740.075351
...,...,...,...,...,...,...,...
775,201912,21263,0.0,-0.717058,0.133198,0.178028,0.005541
776,201912,21265,0.0,1.267023,0.151885,0.136228,0.324488
777,201912,21266,0.0,-0.337602,0.151885,0.134848,0.106360
778,201912,21267,0.0,-5.911647,0.160505,0.073935,-0.276571


Vemos cuantos negativos hay

In [83]:
df_future_copy[df_future_copy['tn'] < 0]

,periodo,product_id,target,pred,tn_mean,tn_std,tn
27,201912,20028,0.0,-7.330667,272.583402,72.774864,-260.904923
142,201912,20168,0.0,-3.042174,55.091778,21.561215,-10.501185
172,201912,20211,0.0,-5.002917,40.769387,11.408079,-16.304291
266,201912,20325,0.0,-1.826853,22.116357,12.551165,-0.812780
307,201912,20379,0.0,-3.550195,16.493350,5.841961,-4.246752
329,201912,20408,0.0,-4.162707,23.317603,7.136667,-6.390249
369,201912,20477,0.0,-4.763937,22.091404,7.804544,-15.088949
379,201912,20491,0.0,-2.167091,19.221130,9.400429,-1.150452
389,201912,20510,0.0,-2.555218,33.025903,13.501399,-1.473119
394,201912,20521,0.0,-2.306823,25.250207,12.197611,-2.887518


Reemplazamos los negativos por el promedio de ultimos 12 meses

In [88]:
promedio780 = model_lgb.promedio_12_meses_780p()
promedio780.rename(columns={"tn":"tn_780"}, inplace=True)
df_future_copy = df_future_copy.merge(promedio780, on='product_id', how='left')
df_future_copy.drop(columns=['target','periodo'], inplace=True)
df_future_copy.loc[df_future_copy['tn'] < 0, 'tn'] = df_future_copy['tn_780']
df_future_copy



,product_id,pred,tn_mean,tn_std,tn,tn_780
0,20001,0.744316,1375.882837,310.489330,1606.984862,1454.732720
1,20002,0.353683,962.547075,258.093361,1053.830220,1175.437142
2,20003,-0.050623,892.114038,307.871182,876.528653,784.976407
3,20004,0.743757,665.386502,225.992372,833.469882,627.215328
4,20005,0.510273,627.744458,220.138757,740.075351,668.270104
...,...,...,...,...,...,...
775,21263,-0.717058,0.133198,0.178028,0.005541,0.029993
776,21265,1.267023,0.151885,0.136228,0.324488,0.089541
777,21266,-0.337602,0.151885,0.134848,0.106360,0.094659
778,21267,-5.911647,0.160505,0.073935,0.092835,0.092835


Reemplazo NaN

In [93]:
promedio780 = model_lgb.promedio_12_meses_780p()
promedio780.rename(columns={"tn":"tn_780"}, inplace=True)
df_future_copy = df_future_copy.merge(promedio780, on='product_id', how='left')
# df_future_copy.drop(columns=['target','periodo'], inplace=True)
df_future_copy.loc[df_future_copy['tn'].isna(), 'tn'] = df_future_copy['tn_780']
df_future_copy

,product_id,pred,tn_mean,tn_std,tn,tn_780_x,tn_780_y,tn_780
0,20001,0.744316,1375.882837,310.489330,1606.984862,1454.732720,1454.732720,1454.732720
1,20002,0.353683,962.547075,258.093361,1053.830220,1175.437142,1175.437142,1175.437142
2,20003,-0.050623,892.114038,307.871182,876.528653,784.976407,784.976407,784.976407
3,20004,0.743757,665.386502,225.992372,833.469882,627.215328,627.215328,627.215328
4,20005,0.510273,627.744458,220.138757,740.075351,668.270104,668.270104,668.270104
...,...,...,...,...,...,...,...,...
775,21263,-0.717058,0.133198,0.178028,0.005541,0.029993,0.029993,0.029993
776,21265,1.267023,0.151885,0.136228,0.324488,0.089541,0.089541,0.089541
777,21266,-0.337602,0.151885,0.134848,0.106360,0.094659,0.094659,0.094659
778,21267,-5.911647,0.160505,0.073935,0.092835,0.092835,0.092835,0.092835


In [94]:
df_future_copy

,product_id,pred,tn_mean,tn_std,tn,tn_780_x,tn_780_y,tn_780
0,20001,0.744316,1375.882837,310.489330,1606.984862,1454.732720,1454.732720,1454.732720
1,20002,0.353683,962.547075,258.093361,1053.830220,1175.437142,1175.437142,1175.437142
2,20003,-0.050623,892.114038,307.871182,876.528653,784.976407,784.976407,784.976407
3,20004,0.743757,665.386502,225.992372,833.469882,627.215328,627.215328,627.215328
4,20005,0.510273,627.744458,220.138757,740.075351,668.270104,668.270104,668.270104
...,...,...,...,...,...,...,...,...
775,21263,-0.717058,0.133198,0.178028,0.005541,0.029993,0.029993,0.029993
776,21265,1.267023,0.151885,0.136228,0.324488,0.089541,0.089541,0.089541
777,21266,-0.337602,0.151885,0.134848,0.106360,0.094659,0.094659,0.094659
778,21267,-5.911647,0.160505,0.073935,0.092835,0.092835,0.092835,0.092835


Guardamos el archivo

In [95]:
# df_future_copy.drop(columns=['tn'], inplace=True)
# df_future_copy.rename(columns={'pred': 'tn'}, inplace=True)
df_future_copy[['product_id', 'tn']].to_csv("./outputs/predicciones_exp_09_lgb_v1.csv", index=False, sep=',')

Ensemble

In [31]:
df_lgb = df_future_copy[['product_id', 'tn']].rename(columns={'tn': 'tn_lgb'})
df_rl = pd.read_csv("./outputs/predicciones_regresion_lineal_v1.csv", sep=',', encoding='utf-8').rename(columns={'tn': 'tn_rl'})
df_ensemble = df_lgb.merge(df_rl, on='product_id', how='left')
df_ensemble['tn'] = (df_ensemble['tn_lgb'] + df_ensemble['tn_rl']) / 2
df_ensemble[['product_id', 'tn']].to_csv("./outputs/predicciones_exp_07_lgb_v6_ensemble.csv", index=False, sep=',')

In [32]:
df_lgb = df_future_copy[['product_id', 'tn']].rename(columns={'tn': 'tn_lgb'})
df_rl = pd.read_csv("./outputs/predicciones_regresion_lineal_v1.csv", sep=',', encoding='utf-8').rename(columns={'tn': 'tn_rl'})
df_ag = pd.read_csv("./outputs/prediccion_autogluon_2ventanas.csv", sep=',', encoding='utf-8').rename(columns={'tn': 'tn_ag'})
df_ensemble = df_lgb.merge(df_rl, on='product_id', how='left')
df_ensemble = df_ensemble.merge(df_ag, on='product_id', how='left')
df_ensemble['tn'] = (df_ensemble['tn_lgb'] + df_ensemble['tn_rl'] + df_ensemble['tn_ag']) / 3
df_ensemble[['product_id', 'tn']].to_csv("./outputs/predicciones_exp_07_lgb_v6_ensemble_3models.csv", index=False, sep=',')